In [ ]:
"""
Shop-Predictor Product Category Classifier - DistilBERT Training
Run this notebook on Google Colab with GPU enabled
"""

!pip install transformers datasets torch pandas scikit-learn accelerate -q


In [ ]:
import torch
import pandas as pd
import numpy as np
from transformers import (
    DistilBertTokenizer,
    DistilBertForSequenceClassification,
    TrainingArguments,
    Trainer,
    pipeline
)
from datasets import Dataset
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import os
import zipfile
from google.colab import files, drive
import joblib
import re

# Check GPU availability
print("GPU Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device:", torch.cuda.get_device_name(0))
    device = torch.device("cuda")
else:
    print("Using CPU - this will be slow!")
    device = torch.device("cpu")

In [ ]:
drive.mount('/content/drive')
dataset_path = '/content/drive/MyDrive/shop_data/'

In [ ]:
dataset_path = '/content/drive/MyDrive/shop_data/'

In [ ]:
os.listdir(dataset_path)

In [ ]:
class CategoryUnifier:
    def __init__(self):
        # Based on your actual data analysis - these are the real mappings we found!
        self.category_mappings = {
            # Wine categories (3,359 total samples)
            'vin': 'wine',           # French (1,946 samples)
            'vino': 'wine',          # Spanish/Italian lowercase (532 samples)
            'Vino': 'wine',          # Spanish/Italian uppercase (881 samples)
            'wine': 'wine',          # English (if any)
            'Wine': 'wine',          # English (if any)
            'Vin': 'wine',          # English (if any)
            'wijn': 'wine',         # Dutch (if any)
            'vinho tinto': 'wine',
            'vinho branco': 'wine',
            'vinho tinto': 'wine',
            'Vinho': 'wine',
            'Spumante': 'wine',
            'Prosecco': 'wine',
            'Vino bianco': 'wine',
            'Vino rosso': 'wine',
            'Lambrusco': 'wine',
            'Chianti': 'wine',
            'Champagne': 'wine',
            'champagne': 'wine',
            "vino blanco": 'wine',
            "vino crianza": 'wine',
            "vino reserva": 'wine',
            "vino rosado": 'wine',
            "vino tinto": 'wine',
            "vino verdejo": 'wine',
            "red wine": 'wine',
            "white wine": 'wine',
            "rose wine": 'wine',


            # Cheese categories (1,903 total samples)
            'fromage': 'cheese',     # French (686 samples),
            'Fromage': 'cheese',     # French (686 samples)
            'formaggio': 'cheese',   # Italian lowercase (648 samples)
            'Formaggio': 'cheese',   # Italian uppercase (648 samples)
            'queso': 'cheese',       # Spanish (569 samples)
            'Queso': 'cheese',       # Spanish (569 samples)
            'cheese': 'cheese',      # English (if any)
            'Cheese': 'cheese',      # English (if any)
            'käse': 'cheese',        # German (if any)
            'queijo': 'cheese',
            'Gorgonzola': 'cheese',
            'Parmigiano': 'cheese',
            'Formaggio Grattugiato': 'cheese',
            'Mozzarella': 'cheese',
            'mozzarella': 'cheese',
            'Certosa': 'cheese',
            'Emmental': 'cheese',
            'Stracchino': 'cheese',
            'Ricotta': 'cheese',
            'Mozzarella di bufala': 'cheese',
            'Pecorino': 'cheese',
            'Sottilette': 'cheese',
            'Asiago': 'cheese',
            'Provolone': 'cheese',
            'Robiola': 'cheese',
            'brie': 'cheese',
            'mascarpone': 'cheese',
            'Mascarpone': 'cheese',
            'Galbanino': 'cheese',
            'Formaggio spalmabile': 'cheese',

            'coca-cola': 'beverages',
            'Coca Cola': 'beverages',
            'Coca-cola': 'beverages',
            'Coca cola': 'beverages',
            'beverages': 'beverages',
            'Beverages': 'beverages',
            'Bebidas': 'beverages',
            'bebidas': 'beverages',
            'boissons': 'beverages',
            'Getränke': 'beverages',
            'Getranke': 'beverages',
            'Bibite': 'beverages',
            'Energy drink': 'beverages',
            'energy drink': 'beverages',
            'Energy Drink': 'beverages',
            'Bevande analcoliche': 'beverages',
            'coca cola zero': 'beverages',
            'Bitter': 'beverages',

            'Lavatrice': 'washing machine',
            'lavatrice': 'washing machine',
            'Washing machine': 'washing machine',
            'washing machine': 'washing machine',

            'Spirits': 'spirits',
            'Liquore': 'spirits',
            'liqueur': 'spirits',
            'spirits': 'spirits',
            'vodka': 'spirits',
            'Vodka': 'spirits',
            'Whisky': 'spirits',
            'whisky': 'spirits',
            'whiskey': 'spirits',
            'Rum': 'spirits',
            'rum': 'spirits',
            'Gin': 'spirits',

            # Games categories (1,417 total samples)
            'juegos': 'games',       # Spanish (764 samples)
            'plush toys': 'games',
            'toys': 'games',
            'jeux': 'games',         # French (653 samples)
            'games': 'games',        # English (if any)
            'giochi': 'games',       # Italian (if any)
            'spiele': 'games',       # German (if any)
            'jogos': 'games',
            'Monopoly': 'games',
            'monopoly': 'games',
            'Giochi per bambini': 'games',
            'peluche': 'games',
            'Peluche': 'games',
            'Puzzle': 'games',
            'puzzle': 'games',
            'Ps5': 'games',
            'PS5': 'games',
            'Nintendo Switch': 'games',
            'Hot Wheels': 'games',
            'Baby Doll': 'games',
            'car games': 'games',
            'console': 'games',
            'Console': 'games',

            # Beer categories (extend as found)
            'cerveza': 'beer',       # Spanish (707 samples)
            'bière': 'beer',         # French (if any)
            'birra': 'beer',         # Italian (if any)
            'beer': 'beer',          # English (if any)
            'bier': 'beer',          # German (if any)
            'cerveja': 'beer',
            'birra moretti': 'beer',
            'birra peroni': 'beer',

            # Pharmacy/Medicine (keep as is, quite international)
            'pharmacy': 'pharmacy',   # (511 samples)
            'Pharmacy': 'pharmacy',   # Case normalization
            'farmacia': 'pharmacy',   # Spanish/Italian
            'pharmacie': 'pharmacy',  # French
            'apotheke': 'pharmacy',   # German
            'produtos farmaceuticos': 'pharmacy',
            'parapharmacie': 'pharmacy',
            'productos farmacéuticos': 'pharmacy',
            'medicines': 'pharmacy',
            "medicamentos": 'pharmacy',
            "medicine": 'pharmacy',

            # Meat categories (if found in data)
            'carne': 'meat',         # Spanish/Italian
            'Carne': 'meat',         # Spanish/Italian
            'steak': 'meat',
            'Steak': 'meat',
            'viande': 'meat',        # French
            'meat': 'meat',          # English
            'fleisch': 'meat',       # German
            'carnes': 'meat',
            'Beef': 'meat',
            'beef': 'meat',
            'Roast Beef': 'meat',
            'roast beef': 'meat',
            'salami': 'meat',
            'Salami': 'meat',
            'Cotolette di pollo': 'meat',
            'pollo': 'meat',
            'Pollo': 'meat',
            'bacon': 'meat',
            'Bacon': 'meat',
            'pancetta': 'meat',
            'salumi': 'meat',
            'Prosciutto crudo': 'meat',
            'Prosciutto cotto': 'meat',
            'Salsicce': 'meat',
            'Hamburger': 'meat',
            'Wurstel': 'meat',
            'Cotoletta': 'meat',
            'Vitello': 'meat',
            'Petto di pollo': 'meat',
            'Bistecca': 'meat',
            'Mortadella': 'meat',
            'Salame': 'meat',
            'Salumi': 'meat',
            'Speck': 'meat',
            'Tacchino': 'meat',
            'Carne macinata': 'meat',
            'cordon bleu': 'meat',
            'Prosciutto di Parma': 'meat',

            'dog food': 'pet food',
            'Dog food': 'pet food',
            'Cibo per cani': 'pet food',
            'cibo per cani': 'pet food',
            'Cibo per gatti': 'pet food',
            'cibo per gatti': 'pet food',
            'cat food': 'pet food',
            'Cat food': 'pet food',
            'Pet care': 'pet food',

            'Books': 'books',
            'books': 'books',
            'libros': 'books',       # Spanish
            'Libros': 'books',       # Spanish
            'livres': 'books',       # French
            'Livres': 'books',       # French
            'Libri': 'books',        # Italian
            'libri': 'books',        # Italian

            'Patatine': 'potatoes',
            'potato chips': 'potatoes',
            'Patatine fritte': 'potatoes',
            'patatine': 'potatoes',
            'Patatas fritas': 'potatoes',
            'patatas fritas': 'potatoes',
            'chips': 'potatoes',
            'Chips': 'potatoes',
            'Pommes frites': 'potatoes',
            'pommes frites': 'potatoes',
            'Kartoffelchips': 'potatoes',
            'kartoffelchips': 'potatoes',
            'Patate': 'potatoes',
            'patate': 'potatoes',
            'Patate surgelate': 'potatoes',

            'Pasta Barilla': 'pasta',
            "Pasta all'uovo": 'pasta',
            'Gnocchi': 'pasta',
            'gnocchi': 'pasta',
            'macarrão': 'pasta',
            'Pasta di semola': 'pasta',
            'Pasta': 'pasta',
            'pasta': 'pasta',
            'pâtes': 'pasta',        # French
            'pasta alimenticia': 'pasta', # Spanish
            'spaghetti': 'pasta',
            'Spaghetti': 'pasta',
            'Penne': 'pasta',
            'Fusilli': 'pasta',
            'Pasta fresca': 'pasta',
            'Lasagne': 'pasta',
            'Tortellini': 'pasta',

            # Bread categories (if found)
            'pain': 'bread',         # French
            'pan': 'bread',          # Spanish
            'pane': 'bread',         # Italian
            'bread': 'bread',        # English
            'brot': 'bread',         # German
            'Pancarrè': 'bread',
            'Panini': 'bread',
            'Piadine': 'bread',
            'Focaccia': 'bread',
            'Pan Bauletto': 'bread',
            'Baguette': 'bread',

            'Alimenti': 'food',
            'food': 'food',
            "alimentación": 'food',
            "alimentation": 'food',
            "alimentazione": 'food',
            "alimentação": 'food',

            'uova': 'eggs',
            'Uova': 'eggs',
            'eggs': 'eggs',
            'Eggs': 'eggs',
            'Oeufs': 'eggs',
            'Ovos': 'eggs',
            'Eier': 'eggs',
            'Huevos': 'eggs',

            'Accessori cucina': 'kitchen accessories',
            "air fryer": 'kitchen accessories',
            "friggitrice": 'kitchen accessories',
            'frigoriferi': 'kitchen accessories',
            'accessoires de cuisine': 'kitchen accessories',
            'accessori casa': 'kitchen accessories',
            'toaster': 'kitchen accessories',
            'fridge': 'kitchen accessories',
            'Electric Oven': 'kitchen accessories',
            'Fridge': 'kitchen accessories',
            'accessori cucina': 'kitchen accessories',
            'utensilios de cocina': 'kitchen accessories',
            'kitchen accessories': 'kitchen accessories',
            'Kitchen Accessories': 'kitchen accessories',
            'Kitchen accessories': 'kitchen accessories',
            'Kettle': 'kitchen accessories',
            'Dishwasher': 'kitchen accessories',
            'Kitchen appliances': 'kitchen accessories',
            'oven': 'kitchen accessories',
            'forno': 'kitchen accessories',

            'Papel Higiénico': 'toilet paper',
            'papel higiénico': 'toilet paper',
            'toilet paper': 'toilet paper',
            'Toilet paper': 'toilet paper',
            'Carta igienica': 'toilet paper',

            'Biscotti': 'biscuits',
            'biscotti': 'biscuits',
            'Oro Saiwa': 'biscuits',
            'Biscuits': 'biscuits',
            'biscuits': 'biscuits',
            'Cookies': 'biscuits',
            'cookies': 'biscuits',
            'Galletas': 'biscuits',
            'Gallette': 'biscuits',

            'Pasticceria': 'pastry',
            'pasticceria': 'pastry',
            'Pastry': 'pastry',
            'pastry': 'pastry',
            'Pâtisserie': 'pastry',
            'pâtisserie': 'pastry',
            'Bäckerei': 'pastry',
            'bäckerei': 'pastry',
            'Bakery': 'pastry',
            'bakery': 'pastry',
            'Panettone': 'pastry',
            'panettone': 'pastry',
            'Pandoro': 'pastry',
            'Cornetti': 'pastry',
            'Caramelle': 'pastry',
            'Dessert': 'pastry',
            'brioche': 'pastry',

            'Profumi': 'perfumes',
            'profumi': 'perfumes',
            'Perfumes': 'perfumes',
            'perfumes': 'perfumes',

            'pizza': 'pizza',
            'Pizza': 'pizza',
            'Pizza Ristorante': 'pizza',
            'pizza Ristorante': 'pizza',

            'kiwi': 'fruit',
            'Arance': 'fruit',
            'Mandarini': 'fruit',
            'Kiwi': 'fruit',
            'oranges': 'fruit',
            'Oranges': 'fruit',
            'frutta': 'fruit',
            'Frutta': 'fruit',
            'fruit': 'fruit',
            'Ananas': 'fruit',
            'Fruit': 'fruit',
            'fruits': 'fruit',
            'Fruits': 'fruit',
            'manzana': 'fruit',
            'Manzana': 'fruit',
            'apple': 'fruit',
            'Apple': 'fruit',
            'banane': 'fruit',
            'bananas': 'fruit',
            'banana': 'fruit',
            'Banane': 'fruit',
            'Mele': 'fruit',
            'Pere': 'fruit',
            'Prugne': 'fruit',

            # Milk/Dairy categories (if found)
            'lait': 'milk',          # French
            'leche': 'milk',         # Spanish
            'latte': 'milk',         # Italian
            'milk': 'milk',          # English
            'milch': 'milk',         # German
            'leite': 'milk',
            'almond milk': 'milk',
            'Latte Granarolo': 'milk',
            'Latte parzialmente scremato': 'milk',
            'Latte intero': 'milk',
            'Latte uht': 'milk',

            # Coffee categories (if found)
            'café': 'coffee',        # French/Spanish
            'Nescafè': 'coffee',
            'caffè': 'coffee',       # Italian
            'coffee': 'coffee',      # English
            'kaffee': 'coffee',      # German
            'cafè': 'coffee',
            'Caffè Kimbo': 'coffee',
            'Caffè': 'coffee',
            'Capsule caffè': 'coffee',

            # Water categories (if found)
            'eau': 'water',          # French
            'agua': 'water',         # Spanish
            'acqua': 'water',        # Italian
            'water': 'water',        # English
            'wasser': 'water',       # German
            'água': 'water',
            'Acqua San Benedetto': 'water',
            "Acqua Sant'Anna": 'water',

            # Oil categories (if found)
            'huile': 'oil',          # French
            'aceite': 'oil',         # Spanish
            'olio': 'oil',           # Italian
            'oil': 'oil',            # English
            'öl': 'oil',             # German
            'Olio di semi': 'oil',
            "olio per friggere": 'oil',
            "olio extravergine di oliva": 'oil',

            # Fish categories (if found)
            'poisson': 'fish',       # French
            'pescado': 'fish',       # Spanish
            'pesce': 'fish',         # Italian
            'orata': 'fish',
            'Alici': 'fish',
            'fish': 'fish',          # English
            'fisch': 'fish',         # German
            'Bastoncini di pesce': 'fish',
            'Tonno': 'fish',
            'Gamberi': 'fish',
            'Filetti di salmone': 'fish',
            'salmon': 'fish',
            'Vongole': 'fish',
            'Baccalà': 'fish',
            'Frutti di mare': 'fish',
            'Tonno Rio mare': 'fish',
            'Salmone affumicato': 'fish',
            'Merluzzo': 'fish',
            'Filetti di sgombro': 'fish',
            'Sgombro': 'fish',
            'Bastoncini Findus': 'fish',
            'Insalata di mare': 'fish',
            'Platessa': 'fish',
            'Filetti di merluzzo': 'fish',
            'Sushi': 'fish',
            'sushi': 'fish',
            'Cozze': 'fish',
            'Trota': 'fish',

            'tablet android': 'tablet',
            'Tablet android': 'tablet',
            'tablet Android': 'tablet',
            'tablet': 'tablet',
            'Tablet': 'tablet',
            'tablet Samsung': 'tablet',
            'tablet samsung': 'tablet',
            'ipad': 'tablet',
            'Ipad': 'tablet',
            'iPad': 'tablet',

            'Samsung Tv': 'tv',
            'Tv': 'tv',
            'TV': 'tv',
            'tv': 'tv',
            'Samsung TV': 'tv',
            'Smart Tv': 'tv',
            'Smart tv': 'tv',
            'smart tv': 'tv',
            'Smart TV': 'tv',
            'Tv 4k': 'tv',
            'Tv 4K': 'tv',
            'tv led': 'tv',
            'Tv led': 'tv',
            'Monitor tv': 'tv',
            'monitor tv': 'tv',
            'monitor': 'tv',
            'Monitor': 'tv',

            'iphone': 'iphone',
            'iPhone': 'iphone',
            'Iphone': 'iphone',
            'IPhone': 'iphone',
            'iPhone': 'iphone',
            'iphone 13': 'iphone',
            'iPhone 13': 'iphone',
            'Iphone 13': 'iphone',
            'Iphone 12': 'iphone',
            'iphone 12': 'iphone',
            'iPhone 12': 'iphone',

            'Smartphone': 'smartphone',
            'smartphone': 'smartphone',
            'Smartphone Samsung': 'smartphone',
            'Smartphone android': 'smartphone',
            'Samsung Galaxy': 'smartphone',
            'smartphones': 'smartphone',

        }

        # Additional case-insensitive mappings for common patterns
        self.case_normalize_first = True

    def normalize_category(self, category):
        """
        Normalize a category name to its unified form
        """
        if not category or pd.isna(category):
            return 'unknown'

        # Convert to string and strip whitespace
        category = str(category).strip()

        # First, try exact match (preserves case sensitivity)
        if category in self.category_mappings:
            return self.category_mappings[category]

        # Then try lowercase match
        category_lower = category.lower()
        if category_lower in self.category_mappings:
            return self.category_mappings[category_lower]

        # If no mapping found, return lowercase normalized version
        return category_lower

    def unify_categories(self, categories):
        """
        Unify a list of categories

        Args:
            categories: List of category names

        Returns:
            List of unified category names
        """
        return [self.normalize_category(cat) for cat in categories]

    def get_mapping_stats(self, categories):
        """
        Get statistics about the unification process

        Args:
            categories: List of original categories

        Returns:
            Dict with unification statistics
        """
        from collections import Counter

        original_counts = Counter(categories)
        unified_categories = self.unify_categories(categories)
        unified_counts = Counter(unified_categories)

        # Find categories that were unified
        unified_groups = {}
        for orig_cat in set(categories):
            unified_cat = self.normalize_category(orig_cat)
            if unified_cat not in unified_groups:
                unified_groups[unified_cat] = []
            unified_groups[unified_cat].append(orig_cat)

        # Find groups that had multiple original categories
        merged_groups = {k: v for k, v in unified_groups.items() if len(v) > 1}

        stats = {
            'original_categories': len(original_counts),
            'unified_categories': len(unified_counts),
            'reduction_count': len(original_counts) - len(unified_counts),
            'reduction_percentage': (len(original_counts) - len(unified_counts)) / len(original_counts) * 100,
            'merged_groups': merged_groups,
            'top_unified_categories': unified_counts.most_common(10)
        }

        return stats

    def print_unification_report(self, categories):
        """Print a detailed unification report"""
        stats = self.get_mapping_stats(categories)

        print("Category Unification Report")
        print("=" * 50)
        print(f"Original categories: {stats['original_categories']:,}")
        print(f"Unified categories: {stats['unified_categories']:,}")
        print(f"Categories merged: {stats['reduction_count']:,} ({stats['reduction_percentage']:.1f}% reduction)")

        print(f"\nTop 10 Unified Categories:")
        for i, (cat, count) in enumerate(stats['top_unified_categories'], 1):
            print(f"  {i:2d}. {cat}: {count:,} samples")

        print(f"\n🔄 Merged Category Groups:")
        for unified_cat, original_cats in stats['merged_groups'].items():
            if len(original_cats) > 1:  # Only show groups that were actually merged
                sample_counts = []
                total_samples = 0
                from collections import Counter
                orig_counts = Counter(categories)

                for orig_cat in original_cats:
                    count = orig_counts[orig_cat]
                    sample_counts.append(f"{orig_cat}({count})")
                    total_samples += count

                print(f"  • {unified_cat}: {' + '.join(sample_counts)} = {total_samples:,} total")


In [ ]:
def clean_text(text):
    """Clean text for better tokenization"""
    if pd.isna(text) or text == "":
        return ""

    text = str(text).lower()
    # Keep more characters for BERT (it handles punctuation well)
    text = re.sub(r'[^\w\s\-\'\.]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def load_and_process_data(directory):
    """Load all uploaded CSV files and process them"""

    # Find all CSV files
    csv_files = [f for f in os.listdir(directory) if f.endswith('.csv')]

    if not csv_files:
        # Try txt files (in case you uploaded the samples as txt)
        csv_files = [f for f in os.listdir(directory) if f.endswith('.txt')]

    print(f"Found {len(csv_files)} data files: {csv_files}")

    # Load all files
    dataframes = []
    for file in csv_files:
        try:
            file = os.path.join(directory, file)
            df = pd.read_csv(file)
            print(f"Loaded {len(df)} rows from {file}")
            dataframes.append(df)
        except Exception as e:
            print(f"Error loading {file}: {e}")

    # Combine all data
    combined_df = pd.concat(dataframes, ignore_index=True)
    print(f"Total combined data: {len(combined_df)} rows")

    # Fill missing values
    combined_df['product_name'] = combined_df['product_name'].fillna('')
    combined_df['product_brand'] = combined_df['product_brand'].fillna('')
    combined_df['category'] = combined_df['category'].fillna('unknown')

    # APPLY CATEGORY UNIFICATION - This is the key improvement!
    print("Applying multilingual category unification...")
    original_categories = combined_df['category'].tolist()
    print(f"Before unification: {combined_df['category'].nunique()} unique categories")

    # Apply unification using the existing CategoryUnifier
    unifier = CategoryUnifier()

    unified_categories = unifier.unify_categories(original_categories)

    print(f"After unification: {len(set(unified_categories))} unique categories")

    # Show the unification impact
    unifier.print_unification_report(original_categories)

    # Create text features (product_name + brand)
    texts = []
    for _, row in combined_df.iterrows():
        text_parts = []
        if row['product_name']:
            text_parts.append(clean_text(row['product_name']))
        if row['product_brand']:
            text_parts.append(clean_text(row['product_brand']))

        combined_text = ' '.join(text_parts)
        texts.append(combined_text if combined_text.strip() else 'unknown product')

    # Clean categories
    # raw_categories = [clean_text(cat) for cat in combined_df['category'].tolist()]
    # raw_categories =


    return texts, unified_categories, combined_df

In [ ]:
print("Loading and processing data...")
texts, categories, df = load_and_process_data(dataset_path)

print(f"Processed {len(texts)} samples")
print(f"Unique categories: {len(set(categories))}")
print("\nSample data:")
for i in range(3):
    print(f"Text: {texts[i][:100]}...")
    print(f"Category: {categories[i]}")
    print("---")

# Get category distribution
from collections import Counter
category_counts = Counter(categories)
print("\nTop 10 categories:")
for cat, count in category_counts.most_common(10):
    print(f"{cat}: {count}")

In [ ]:
# 2. Filter rare categories
from collections import Counter
unified_categories = categories
MIN_SAMPLES_PER_CATEGORY = 20
counts = Counter(unified_categories)
valid_idx = [i for i, cat in enumerate(unified_categories) if counts[cat] >= MIN_SAMPLES_PER_CATEGORY]

texts = [texts[i] for i in valid_idx]
print(f"Filtered dataset: {len(texts)} samples")
filtered_categories = [unified_categories[i] for i in valid_idx]

# 3. Encode labels
label_encoder = LabelEncoder()
encoded_labels = label_encoder.fit_transform(filtered_categories)
num_labels = len(label_encoder.classes_)


print(f"Number of unique categories: {num_labels}")


# Split data
train_texts, test_texts, train_labels, test_labels = train_test_split(
    texts, encoded_labels, test_size=0.2, random_state=42, stratify=encoded_labels
)

print(f"Training samples: {len(train_texts)}")
print(f"Test samples: {len(test_texts)}")

# Initialize tokenizer
model_name = "distilbert-base-multilingual-cased"  # Works for multiple languages
tokenizer = DistilBertTokenizer.from_pretrained(model_name)

# Tokenize data
print("Tokenizing data...")
train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=128)
test_encodings = tokenizer(test_texts, truncation=True, padding=True, max_length=128)

# Create datasets
train_dataset = Dataset.from_dict({
    'input_ids': train_encodings['input_ids'],
    'attention_mask': train_encodings['attention_mask'],
    'labels': train_labels.tolist()
})

test_dataset = Dataset.from_dict({
    'input_ids': test_encodings['input_ids'],
    'attention_mask': test_encodings['attention_mask'],
    'labels': test_labels.tolist()
})

print("Data preparation complete!")


In [ ]:
pip install --upgrade transformers

In [ ]:
import transformers
print(transformers.__version__)

In [ ]:
# Initialize model
model = DistilBertForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels
)

# Move to GPU if available
model = model.to(device)

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=100,
    # evaluation_strategy="epoch",  # <-- must match save_strategy
    save_strategy="epoch",
    # load_best_model_at_end=True
)

# Metrics function
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return {'accuracy': accuracy_score(labels, predictions)}

# Initialize trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

print("Starting training...")
print("This will take approximately 20-30 minutes on GPU")

# Train the model
trainer.train()

print("Training completed!")

In [ ]:
print("Evaluating model on test set...")

# Get predictions
predictions = trainer.predict(test_dataset)
y_pred = np.argmax(predictions.predictions, axis=1)

# Calculate metrics
accuracy = accuracy_score(test_labels, y_pred)
print(f"Test Accuracy: {accuracy:.4f}")

# Detailed classification report
target_names = [str(label) for label in label_encoder.classes_]
report = classification_report(test_labels, y_pred, target_names=target_names)
print("Classification Report:")
print(report)

# Test some predictions
print("\nSample predictions:")
test_samples = [
    "iPhone 13 Apple",
    "Lavatrastes Limón Nice Kleen",
    "Nurofen Medicine Children",
    "Coca Cola drink"
]

for sample in test_samples:
    inputs = tokenizer(sample, return_tensors="pt", truncation=True, padding=True, max_length=128)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        predictions = torch.nn.functional.softmax(outputs.logits, dim=-1)
        predicted_class = torch.argmax(predictions, dim=-1).item()
        confidence = predictions[0][predicted_class].item()

    predicted_category = label_encoder.inverse_transform([predicted_class])[0]
    print(f"Text: {sample}")
    print(f"Predicted: {predicted_category} (confidence: {confidence:.4f})")
    print("---")

In [ ]:
print("Saving model and tokenizer...")

# Create directories
os.makedirs("shop_distilbert_model", exist_ok=True)

# Save model and tokenizer
model.save_pretrained("shop_distilbert_model")
tokenizer.save_pretrained("shop_distilbert_model")

# Save label encoder
joblib.dump(label_encoder, "shop_distilbert_model/label_encoder.joblib")

# Save model info
model_info = {
    'accuracy': accuracy,
    'num_categories': num_labels,
    'model_name': model_name,
    'categories': label_encoder.classes_.tolist()
}

import json
with open("shop_distilbert_model/model_info.json", "w") as f:
    json.dump(model_info, f, indent=2)

print("Model saved successfully!")

In [ ]:
# Zip the model for easy download
import shutil

print("Creating downloadable package...")
shutil.make_archive("shop_distilbert_model", "zip", "shop_distilbert_model")

# Download the model
files.download("shop_distilbert_model.zip")

print("Model package downloaded!")
print("\nTo use locally:")
print("1. Extract the zip file to your local project's 'models/' directory")
print("2. Use the local integration code provided")

In [ ]:
# Create a simple prediction pipeline for final testing
classifier = pipeline(
    "text-classification",
    model="shop_distilbert_model",
    tokenizer="shop_distilbert_model",
    device=0 if torch.cuda.is_available() else -1
)

print("Testing final pipeline:")
test_texts = [
    "Samsung Galaxy smartphone",
    "Organic olive oil extra virgin",
    "Children's medicine paracetamol"
]

for text in test_texts:
    result = classifier(text)
    numeric_label = int(result[0]['label'].split('_')[1])  # LABEL_815 -> 815
    category_name = label_encoder.inverse_transform([numeric_label])[0]
    score = result[0]['score']
    print(f"Text: {text}")
    print(f"Predicted category: {category_name}, confidence: {score}")
    print("---")
# Option 2

# for text in test_texts:
#     result = classifier(text)
#     print(f"Text: {text}")
#     print(f"Result: {result}")
#     print("---")

print("Notebook complete! Your model is ready for local use.")
